# AI Video Automation Agent - Pipeline Test

This notebook tests the full AI video generation pipeline (except video rendering and lip sync).

**What this notebook does:**
1. Installs required libraries
2. Tests your Gemini API key
3. Generates a script/screenplay from your idea
4. Creates visual prompts for each scene
5. Generates voiceover audio using Edge TTS
6. Displays all results
7. Provides download links

**Requirements:** Google Gemini API Key (get one free at https://aistudio.google.com/apikey)

## Cell 1: Setup - Install Required Libraries

In [ ]:
# Install required packages
!pip install -q google-generativeai edge-tts moviepy

import os
import json
import asyncio
import nest_asyncio
from pathlib import Path
from IPython.display import display, HTML, Audio, Markdown

# Allow nested async calls in Colab
nest_asyncio.apply()

# Create output directory
OUTPUT_DIR = Path('/content/video_automation_output')
OUTPUT_DIR.mkdir(exist_ok=True)

print('Setup complete! All libraries installed.')
print(f'Output directory: {OUTPUT_DIR}')

## Cell 2: Configuration - Enter Your API Key and Settings

In [ ]:
# ===== CONFIGURATION =====
# Enter your Gemini API key below
GEMINI_API_KEY = ''  # <-- Paste your API key here (starts with AIza...)

# Video generation settings
VIDEO_IDEA = 'A romantic Hindi love song about monsoon rain in Mumbai, showing a couple meeting at Marine Drive during sunset'
LANGUAGE = 'Hindi'  # Options: Hindi, English, Hinglish
MUSIC_TYPE = 'song'  # Options: song, voiceover
VIDEO_STYLE = 'realistic'  # Options: realistic, animated, mixed

# TTS Voice (for Edge TTS)
TTS_VOICE = 'hi-IN-SwaraNeural'  # Hindi female voice
# Other options: 'hi-IN-MadhurNeural' (Hindi male), 'en-US-JennyNeural' (English female)

print(f'Idea: {VIDEO_IDEA}')
print(f'Language: {LANGUAGE}')
print(f'Music Type: {MUSIC_TYPE}')
print(f'Video Style: {VIDEO_STYLE}')
print(f'Voice: {TTS_VOICE}')

## Cell 3: Test Gemini Connection - Validate API Key

In [ ]:
import google.generativeai as genai

if not GEMINI_API_KEY or GEMINI_API_KEY == '':
    print('ERROR: Please enter your Gemini API key in Cell 2 above!')
    raise ValueError('API key is required')

# Configure Gemini
genai.configure(api_key=GEMINI_API_KEY)

# Test the connection
try:
    model = genai.GenerativeModel('gemini-1.5-flash')
    response = model.generate_content('Say hello in one word.')
    print(f'API Key is valid!')
    print(f'Test response: {response.text}')
    print('\nGemini connection successful. Ready to generate!')
except Exception as e:
    print(f'ERROR: API key validation failed: {e}')
    print('Please check your API key and try again.')
    raise

## Cell 4: Script Generation - Generate Screenplay from Your Idea

In [ ]:
# Script generation prompt (same logic as backend/agents/script_agent.py)
SCRIPT_PROMPT = f"""You are a professional music video director and songwriter.
Create a complete screenplay for a {MUSIC_TYPE} video based on this idea:

IDEA: {VIDEO_IDEA}

Language: {LANGUAGE}
Style: {VIDEO_STYLE}
Duration: 3-5 minutes

Generate a JSON response with this EXACT structure:
{{
    "title": "Song/Video Title",
    "synopsis": "Brief 2-3 line synopsis",
    "mood": "Overall mood/emotion",
    "color_palette": ["color1", "color2", "color3"],
    "scenes": [
        {{
            "scene_number": 1,
            "scene_type": "intro/verse/chorus/bridge/outro",
            "description": "Detailed scene description",
            "visual_prompt": "Detailed visual description for image generation",
            "dialogue": "Lyrics or dialogue for this scene",
            "duration_seconds": 15,
            "camera_movement": "pan/zoom/static/tracking",
            "mood": "Scene mood"
        }}
    ]
}}

Create 8-12 scenes covering the full song structure.
IMPORTANT: Return ONLY valid JSON, no markdown formatting."""

print('Generating screenplay...')
model = genai.GenerativeModel('gemini-1.5-flash')
response = model.generate_content(SCRIPT_PROMPT)

# Parse the response
try:
    # Clean up response (remove markdown code blocks if present)
    text = response.text.strip()
    if text.startswith('```json'):
        text = text[7:]
    if text.startswith('```'):
        text = text[3:]
    if text.endswith('```'):
        text = text[:-3]
    text = text.strip()

    screenplay = json.loads(text)
    
    # Save screenplay
    screenplay_path = OUTPUT_DIR / 'screenplay.json'
    with open(screenplay_path, 'w', encoding='utf-8') as f:
        json.dump(screenplay, f, ensure_ascii=False, indent=2)
    
    print(f'\nTitle: {screenplay.get("title", "Untitled")}')
    print(f'Synopsis: {screenplay.get("synopsis", "")}')
    print(f'Mood: {screenplay.get("mood", "")}')
    print(f'Scenes: {len(screenplay.get("scenes", []))}')
    print(f'\nScreenplay saved to: {screenplay_path}')
    
    # Display scenes
    for scene in screenplay.get('scenes', []):
        print(f'\n--- Scene {scene["scene_number"]}: {scene.get("scene_type", "")} ---')
        print(f'  Description: {scene.get("description", "")[:100]}...')
        if scene.get('dialogue'):
            print(f'  Lyrics: {scene["dialogue"][:80]}...')
        print(f'  Duration: {scene.get("duration_seconds", 0)}s')

except json.JSONDecodeError as e:
    print(f'Warning: Could not parse JSON response. Saving raw text.')
    print(f'Parse error: {e}')
    screenplay = {'raw_text': response.text, 'scenes': []}
    with open(OUTPUT_DIR / 'screenplay_raw.txt', 'w', encoding='utf-8') as f:
        f.write(response.text)
    print(f'Raw response saved.')

## Cell 5: Visual Prompt Generation - Create Image/Video Prompts for Each Scene

In [ ]:
# Generate enhanced visual prompts (same logic as backend/agents/visual_agent.py)
VISUAL_PROMPT_TEMPLATE = """You are a professional cinematographer and visual effects artist.
For each scene in this screenplay, create detailed prompts for AI image/video generation.

SCREENPLAY:
{screenplay_json}

For EACH scene, generate:
{{
    "visual_prompts": [
        {{
            "scene_number": 1,
            "image_prompt": "Highly detailed prompt for Imagen/DALL-E (include style, lighting, colors, composition)",
            "video_prompt": "Prompt for Veo/Runway video generation (include motion, transitions)",
            "negative_prompt": "Things to avoid in generation",
            "camera_direction": "Specific camera movement instructions",
            "duration_seconds": 15,
            "aspect_ratio": "16:9"
        }}
    ]
}}

Style: {style}
Make prompts extremely detailed and specific.
IMPORTANT: Return ONLY valid JSON, no markdown formatting."""

scenes = screenplay.get('scenes', [])
if not scenes:
    print('No scenes found in screenplay. Please run Cell 4 first.')
else:
    prompt = VISUAL_PROMPT_TEMPLATE.format(
        screenplay_json=json.dumps(screenplay, ensure_ascii=False, indent=2)[:3000],
        style=VIDEO_STYLE
    )
    
    print(f'Generating visual prompts for {len(scenes)} scenes...')
    response = model.generate_content(prompt)
    
    try:
        text = response.text.strip()
        if text.startswith('```json'):
            text = text[7:]
        if text.startswith('```'):
            text = text[3:]
        if text.endswith('```'):
            text = text[:-3]
        text = text.strip()
        
        visual_data = json.loads(text)
        visual_prompts = visual_data.get('visual_prompts', [])
        
        # Save visual prompts
        visual_path = OUTPUT_DIR / 'visual_prompts.json'
        with open(visual_path, 'w', encoding='utf-8') as f:
            json.dump(visual_data, f, ensure_ascii=False, indent=2)
        
        print(f'\nGenerated {len(visual_prompts)} visual prompts!')
        print(f'Saved to: {visual_path}')
        
        for vp in visual_prompts:
            print(f'\n--- Scene {vp.get("scene_number", "?")} ---')
            print(f'  Image: {vp.get("image_prompt", "")[:120]}...')
            print(f'  Video: {vp.get("video_prompt", "")[:120]}...')
            print(f'  Camera: {vp.get("camera_direction", "")}')
            
    except json.JSONDecodeError as e:
        print(f'Warning: Could not parse visual prompts JSON: {e}')
        visual_prompts = []
        with open(OUTPUT_DIR / 'visual_prompts_raw.txt', 'w', encoding='utf-8') as f:
            f.write(response.text)

## Cell 6: Audio Generation - Generate Voiceover with Edge TTS

In [ ]:
import edge_tts

# Create audio output directory
audio_dir = OUTPUT_DIR / 'audio'
audio_dir.mkdir(exist_ok=True)

async def generate_audio_for_scenes(scenes, voice, output_dir):
    """Generate TTS audio for each scene's dialogue/lyrics."""
    results = []
    
    for scene in scenes:
        scene_num = scene.get('scene_number', 0)
        dialogue = scene.get('dialogue', '') or scene.get('description', '')
        
        if not dialogue:
            print(f'  Scene {scene_num}: No dialogue, skipping.')
            results.append({'scene_number': scene_num, 'status': 'skipped', 'reason': 'no dialogue'})
            continue
        
        output_file = output_dir / f'scene_{str(scene_num).zfill(2)}.mp3'
        
        try:
            communicate = edge_tts.Communicate(dialogue, voice)
            await communicate.save(str(output_file))
            
            # Get file size
            size_kb = output_file.stat().st_size / 1024
            
            print(f'  Scene {scene_num}: Generated ({size_kb:.1f} KB)')
            results.append({
                'scene_number': scene_num,
                'status': 'completed',
                'file': str(output_file),
                'size_kb': round(size_kb, 1)
            })
        except Exception as e:
            print(f'  Scene {scene_num}: ERROR - {e}')
            results.append({'scene_number': scene_num, 'status': 'failed', 'error': str(e)})
    
    return results

# Generate audio
scenes = screenplay.get('scenes', [])
if not scenes:
    print('No scenes found. Please run Cell 4 first.')
else:
    print(f'Generating audio for {len(scenes)} scenes using voice: {TTS_VOICE}')
    print()
    
    audio_results = asyncio.get_event_loop().run_until_complete(
        generate_audio_for_scenes(scenes, TTS_VOICE, audio_dir)
    )
    
    # Summary
    completed = [r for r in audio_results if r['status'] == 'completed']
    print(f'\nAudio generation complete!')
    print(f'  Completed: {len(completed)}/{len(scenes)} scenes')
    
    # Save results
    with open(OUTPUT_DIR / 'audio_results.json', 'w') as f:
        json.dump(audio_results, f, indent=2)

## Cell 7: Display Results - View All Generated Content

In [ ]:
from IPython.display import display, HTML, Audio, Markdown

# Display screenplay
display(Markdown('# Generated Results'))
display(Markdown('---'))

# Title and Synopsis
display(Markdown(f'## {screenplay.get("title", "Untitled")}'))
display(Markdown(f'*{screenplay.get("synopsis", "")}*'))
display(Markdown(f'**Mood:** {screenplay.get("mood", "")} | **Scenes:** {len(screenplay.get("scenes", []))}'))
display(Markdown('---'))

# Display each scene
display(Markdown('## Scenes'))
for scene in screenplay.get('scenes', []):
    scene_md = f"""### Scene {scene['scene_number']}: {scene.get('scene_type', '').title()}
**Description:** {scene.get('description', '')}

**Lyrics/Dialogue:** {scene.get('dialogue', 'N/A')}

**Visual:** {scene.get('visual_prompt', '')}

**Duration:** {scene.get('duration_seconds', 0)}s | **Camera:** {scene.get('camera_movement', 'static')}

---"""
    display(Markdown(scene_md))

# Display audio files
display(Markdown('## Generated Audio Files'))
audio_files = sorted(audio_dir.glob('*.mp3'))
if audio_files:
    for af in audio_files:
        display(Markdown(f'**{af.name}** ({af.stat().st_size / 1024:.1f} KB)'))
        display(Audio(str(af)))
else:
    print('No audio files generated.')

# Summary
display(Markdown('---'))
display(Markdown('## Summary'))
total_duration = sum(s.get('duration_seconds', 0) for s in screenplay.get('scenes', []))
display(Markdown(f'- **Total Duration:** ~{total_duration}s ({total_duration/60:.1f} min)'))
display(Markdown(f'- **Audio Files:** {len(audio_files)}'))
display(Markdown(f'- **Output Directory:** `{OUTPUT_DIR}`'))

## Cell 8: Download Results - Get All Generated Files

In [ ]:
import shutil
from google.colab import files

# Create a zip file with all outputs
zip_path = '/content/video_automation_results'
shutil.make_archive(zip_path, 'zip', str(OUTPUT_DIR))

print('All results packaged!')
print(f'\nFiles included:')
for f in sorted(OUTPUT_DIR.rglob('*')):
    if f.is_file():
        print(f'  - {f.relative_to(OUTPUT_DIR)} ({f.stat().st_size / 1024:.1f} KB)')

print(f'\nTotal package size: {Path(zip_path + ".zip").stat().st_size / 1024:.1f} KB')
print()

# Download
print('Starting download...')
files.download(zip_path + '.zip')
print('\nDownload complete! Extract the zip to view all generated files.')